# Introduction

In this notebook we will try to apply the newest Google's Perch model to identify the birds, insects, mammals and amphibians. Unfortunately, this model is too slow, and it was impossible (at least for me, but you can do better) to make the predictions for all the provided files. So, only a part of them is processed. This makes it possible to estimate the overall score of the model.

# Import common libraries

Note: we need tensorflow-2.20 at least, and tensorflow-2.19 that is default in kaggle environment for now, produces error!

In [ ]:
!pip install /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl
!pip install /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

In [ ]:
import time
START = time.time()

from concurrent.futures import ThreadPoolExecutor
import glob
import librosa
import numpy as np
import os
import pandas as pd
import re
import sys
import tensorflow as tf

tf.experimental.numpy.experimental_enable_numpy_behavior()

TERMINATE_TIME = START + 5300

# Import the model
birdclassifier = tf.saved_model.load('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1/')

# Species mapping

Perch is intended to predict *a lot of* species, while we are working with just few of them here. It is important to get the predictions in the right order.

In [ ]:
bc_labels = pd.read_csv('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1/assets/labels.csv')
print(bc_labels)

We can see, that Perch's labels are in scientific format. So we need a common name for each of the species we work with. Luckly, we have taxonomy data.

In [ ]:
bc_labels = bc_labels.reset_index().rename({'inat2024_fsd50k': 'scientific_name', 'index': 'bc_index'}, axis=1).set_index('scientific_name')
taxonomy = pd.read_csv('/kaggle/input/competitions/birdclef-2026/taxonomy.csv')

mapping = taxonomy.join(bc_labels, on='scientific_name', how='left')
mapping.bc_index = mapping.bc_index.fillna(len(bc_labels)).astype(int)
mapping = mapping[['primary_label', 'bc_index']].set_index('primary_label')

primary_labels = pd.read_csv('/kaggle/input/competitions/birdclef-2026/sample_submission.csv').columns[1:].to_list()
birdclassifier_indices = [int(mapping.loc[pl][0]) for pl in primary_labels]
birdclassifier_indices[:5]

In [ ]:
total_predicted_species = len(set(birdclassifier_indices) - {len(bc_labels)})
print(f'Note: we can predict {total_predicted_species} out of 234 species only!')

# Get the data files

In [ ]:
def get_oggs(max_oggs=8):
    if len(glob.glob('/kaggle/input/competitions/birdclef-2026/test_soundscapes/*.ogg')) > 0:
        oggs = glob.glob('/kaggle/input/competitions/birdclef-2026/test_soundscapes/*.ogg')
    else:
        oggs = sorted(glob.glob(f'/kaggle/input/competitions/birdclef-2026/train_soundscapes/*.ogg'))[:max_oggs]
    return [(n, ogg, re.search(r'/([^/]+)\.ogg$', ogg).group(1), n < int(len(oggs))) for n, ogg in enumerate(oggs)]

oggs = get_oggs()

# Process the files in threads

In [ ]:
def perch_result(ogg):
    _, fname, ss_id, do_prediction = ogg
    sr = 32_000

    print(f'{ss_id}')
    row_ids = [f'{ss_id}_{n}' for n in range(5, 65, 5)]

    if not do_prediction or time.time() > TERMINATE_TIME:
        return row_ids, -1000 * np.ones((12, len(primary_labels)))
        
    data, _ = librosa.load(fname, sr=sr)
    model_outputs = birdclassifier.signatures['serving_default'](inputs=data.reshape((-1, 5 * sr)))['label']

    model_outputs = tf.pad(model_outputs, tf.constant([[0, 0], [0, 1]]))
    result = model_outputs[:, birdclassifier_indices]

    return row_ids, result

In [ ]:
row_ids = []
result = []

with ThreadPoolExecutor(max_workers=4) as executor:
    for ogg_row_ids, ogg_result in executor.map(perch_result, oggs):
        row_ids += ogg_row_ids
        result.append(ogg_result)

submission = pd.DataFrame(np.concatenate(result), columns=primary_labels)
submission['row_id'] = row_ids
submission = submission[['row_id'] + primary_labels]

# Make a submission

In [ ]:
submission.to_csv('submission.csv', index=False)

# Display submission DataFrame
display(submission.head(20))

display(submission.tail(20))